# TensorFlow 도형 분류 실습

이 노트북은 TensorFlow/Keras로 작은 CNN(합성곱 신경망)을 직접 학습시키고, 사용자가 그린 도형 이미지를 분류하는 입문 실습입니다.

**사용법:** Google Colab에서 노트북을 연 뒤 셀을 위에서부터 순서대로 실행하세요. 마지막 단계에서는 그림판에서 검은 선으로 도형을 그리고 PNG로 저장해 업로드합니다.

**분류 대상:** 원, 타원, 정사각형, 사다리꼴, 이등변삼각형, 일반 삼각형.

참고: 학습 이미지는 코드로 합성합니다. 이 실습은 학습 원리를 체험하기 위한 소규모 예제이며, 실제 손그림에서 높은 정확도를 보장하지 않습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import tensorflow as tf

print('TensorFlow 버전:', tf.__version__)
print('사용 장치:', tf.config.list_physical_devices())

## 1. 학습용 도형 이미지 만들기

모델은 도형의 이름을 미리 아는 것이 아닙니다. 각 이미지와 정답 라벨을 여러 개 보여 주면, 이미지의 패턴과 라벨 사이의 관계를 학습합니다. 아래 코드는 위치·크기·회전에 약간의 변화를 주어 합성 데이터를 생성합니다.

In [ ]:
CLASS_NAMES = ['원', '타원', '정사각형', '사다리꼴', '이등변삼각형', '일반 삼각형']
IMG_SIZE = 64

def make_shape(label, rng, size=IMG_SIZE):
    # 흰 배경에 검은 선으로 도형을 그립니다.
    img = Image.new('L', (size, size), 255)
    draw = ImageDraw.Draw(img)
    cx = size / 2 + rng.randint(-4, 4)
    cy = size / 2 + rng.randint(-4, 4)
    w = rng.randint(27, 40)
    h = rng.randint(27, 40)
    stroke = rng.choice([2, 3, 4])

    if label == 0:  # 원
        d = rng.randint(29, 37)
        box = [cx-d/2, cy-d/2, cx+d/2, cy+d/2]
        draw.ellipse(box, outline=0, width=stroke)
    elif label == 1:  # 타원
        box = [cx-w/2, cy-h/2, cx+w/2, cy+h/2]
        # 원과 구분되도록 가로·세로 비율이 달라지게 합니다.
        if abs(w-h) < 7:
            w = min(43, w + 10)
            box = [cx-w/2, cy-h/2, cx+w/2, cy+h/2]
        draw.ellipse(box, outline=0, width=stroke)
    elif label == 2:  # 정사각형
        side = rng.randint(29, 39)
        box = [cx-side/2, cy-side/2, cx+side/2, cy+side/2]
        draw.rectangle(box, outline=0, width=stroke)
    elif label == 3:  # 사다리꼴
        top = rng.randint(13, 23)
        bottom = rng.randint(34, 43)
        points = [(cx-top/2, cy-h/2), (cx+top/2, cy-h/2),
                  (cx+bottom/2, cy+h/2), (cx-bottom/2, cy+h/2)]
        draw.polygon(points, outline=0, fill=None)
        draw.line(points + [points[0]], fill=0, width=stroke)
    elif label == 4:  # 이등변삼각형
        half = rng.randint(15, 21)
        points = [(cx, cy-h/2), (cx+half, cy+h/2), (cx-half, cy+h/2)]
        draw.line(points + [points[0]], fill=0, width=stroke)
    else:  # 일반 삼각형: 세 변이 대체로 다른 형태
        points = [(cx+rng.randint(-15, 10), cy-h/2),
                  (cx+rng.randint(8, 21), cy+h/2),
                  (cx-rng.randint(10, 22), cy+h/2-rng.randint(-3, 8))]
        draw.line(points + [points[0]], fill=0, width=stroke)

    # 약한 회전으로 손그림의 방향 변화 일부를 흉내 냅니다.
    angle = rng.uniform(-12, 12)
    img = img.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=255)
    return np.asarray(img, dtype=np.float32) / 255.0

rng = np.random.default_rng(42)
py_rng = __import__('random').Random(42)
N_PER_CLASS = 1000
images, labels = [], []
for class_id in range(len(CLASS_NAMES)):
    for _ in range(N_PER_CLASS):
        images.append(make_shape(class_id, py_rng))
        labels.append(class_id)

X = np.array(images, dtype=np.float32)[..., np.newaxis]
y = np.array(labels, dtype=np.int64)
indices = rng.permutation(len(y))
X, y = X[indices], y[indices]
split = int(len(y) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print('학습 이미지:', X_train.shape, '테스트 이미지:', X_test.shape)

In [ ]:
# 학습 이미지 예시 확인
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(CLASS_NAMES[y_train[i]])
    ax.axis('off')
plt.tight_layout()

## 2. CNN 모델 구성

합성곱 층은 이미지의 선·모서리 같은 지역 패턴을 학습하고, 마지막 Dense 층은 각 도형 클래스에 대한 점수를 출력합니다. `softmax`는 이 점수들을 합이 1인 값으로 바꿉니다.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    tf.keras.layers.Conv2D(16, kernel_size=3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, kernel_size=3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 3. 모델 학습

각 epoch마다 모델은 학습 데이터를 여러 번 살펴보고, 손실을 줄이도록 가중치를 갱신합니다. `validation_split`은 학습 데이터 일부를 검증에 사용합니다.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=8,
    batch_size=64,
    validation_split=0.15,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'테스트 손실: {test_loss:.4f}')
print(f'테스트 정확도: {test_acc:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()

## 4. 직접 그린 도형 분류하기

1. Windows 그림판을 열고 흰 배경에 검은색 선으로 도형 하나를 크게 그립니다.
2. PNG 파일로 저장합니다.
3. 아래 셀을 실행하고 파일을 업로드합니다.

입력 이미지는 흑백으로 바꾸고, 정사각형으로 맞춘 뒤 64×64로 축소합니다. 실제 손그림은 합성 학습 데이터와 차이가 있으므로 결과가 틀릴 수 있습니다.

In [ ]:
from google.colab import files
uploaded = files.upload()  # PNG/JPG 그림 파일 선택
filename = next(iter(uploaded))

user_img = Image.open(filename).convert('L')
# 비율을 유지하며 흰색 정사각형 캔버스 중앙에 배치
user_img.thumbnail((52, 52))
canvas = Image.new('L', (64, 64), 255)
canvas.paste(user_img, ((64-user_img.width)//2, (64-user_img.height)//2))
arr = np.asarray(canvas, dtype=np.float32) / 255.0
plt.imshow(arr, cmap='gray', vmin=0, vmax=1)
plt.title('전처리된 입력 이미지')
plt.axis('off');

probs = model.predict(arr[np.newaxis, ..., np.newaxis], verbose=0)[0]
order = np.argsort(probs)[::-1]
print('예측 결과:', CLASS_NAMES[int(order[0])])
for idx in order:
    print(f'{CLASS_NAMES[int(idx)]:<10} {probs[idx]:.1%}')

plt.figure(figsize=(7, 3))
plt.bar([CLASS_NAMES[i] for i in range(len(CLASS_NAMES))], probs)
plt.ylabel('모델 출력값')
plt.xticks(rotation=25)
plt.tight_layout()

## 5. 실습 후 탐구할 질문

1. `epochs`를 8에서 15로 늘리면 테스트 정확도가 어떻게 달라지는가?
2. `learning_rate`를 0.001에서 0.01로 바꾸면 학습 곡선이 어떻게 변하는가?
3. `Conv2D` 층을 하나 제거하면 성능에 어떤 영향이 있는가?
4. 원과 타원을 모델이 자주 혼동한다면 학습 데이터를 어떻게 개선할 수 있는가?

이 실습의 분류 확률은 모델의 점수이지, 실제 정답일 확률이 완벽하게 보정되었다는 뜻은 아닙니다.